In [1]:
import pandas as pd

# Load your Excel file
df = pd.read_excel("/nfs/turbo/umms-atjanke/liuwent/Schema/20260415/pe-schema.xlsx",sheet_name="Abstraction-Schema")  # adjust path

print("Available variables to choose from:\n")
for _, row in df.iterrows():
    print(f"- {row['Variable Name']} ({row['Type']}): {row['Instructions']}")

Available variables to choose from:

- shortness_of_breath (presence): Does the note indicate the patient is complaining about shortness of breath?
- chest_pain (presence): Does the note indicate the patient is complaining about chest pain?
- pleuritic_pain (presence): Does the note indicate that there is a 'pleuritic' pain (a chest, back, or other thoracic or truncal pain that is explicitly worse with breathing)? If the patient does *not* have any pain complaint, then mark 'explicitly absent.'
- back_pain (presence): Does the note indicate the patient is complaining about back pain?
- cough (presence): Does the note indicate the patient is complaining of cough?
- hemoptysis_present (presence): Look for any mention of the patient coughing up blood. Include synonyms like “bloody sputum” or “blood-tinged mucus.” If not described, mark as “not mentioned.” If cough is 'explicitly absent', then mark hemotypsis as 'explicitly absent.'
- syncope (presence): Does the note indicate syncope or p

In [2]:
# Pick the variables you want (edit this list)
selected_vars = [
    # Symptom presence
    "shortness_of_breath",
    "chest_pain",
    "pleuritic_pain",
    "back_pain",
    "cough",
    "hemoptysis_present",
    "syncope",
    "exertional_symptoms",

    # Physical exam / history findings
    "unilateral_leg_swelling_present",
    "prior_dvt_pe_present",

    # Risk factors
    "recent_surgery",
    "recent_immobilization",
    "recent_travel",
    "recent_or_active_malignancy",
    "estrogen_use_present",
    "pregnancy",

    # Decision-rule mentions
    "perc_mentioned",
    "wells_mentioned",
]

# Filter rows
df_sel = df[df["Variable Name"].isin(selected_vars)]

# Build the --var commands
var_flags = []
for _, row in df_sel.iterrows():
    var_flags.append(
        f'--var "{row["Variable Name"]}:{row["Type"]}:{row["Instructions"]}"'
    )

var_str = " \\\n  ".join([
    f'--var "{row["Variable Name"]}:{row["Type"]}:{row["Instructions"]}"'
    for _, row in df_sel.iterrows()
])

cmd = f"""python batch-abstract-notes-logged.py \\
  --input ./notes-for-200-cases.csv \\
  --output ./notes-for-2-cases-18features-gpt5-nano-test.parquet \\
  --note-col Text \\
  --id-col EncounterCsn \\
  --script ./llm-chart-abstraction-call.py \\
  {var_str} \\
  --exp-all \\
  --quote-per-var \\
  --repair \\
  --rps 2 \\
  --checkpoint-every 10 \\
  --model gpt-5-nano \\
  --max-rows 2 \\
  --json-out ./debug_nano_test.json    
"""

print(cmd)

python batch-abstract-notes-logged.py \
  --input ./notes-for-200-cases.csv \
  --output ./notes-for-2-cases-18features-gpt5-nano-test.parquet \
  --note-col Text \
  --id-col EncounterCsn \
  --script ./llm-chart-abstraction-call.py \
  --var "shortness_of_breath:presence:Does the note indicate the patient is complaining about shortness of breath?" \
  --var "chest_pain:presence:Does the note indicate the patient is complaining about chest pain?" \
  --var "pleuritic_pain:presence:Does the note indicate that there is a 'pleuritic' pain (a chest, back, or other thoracic or truncal pain that is explicitly worse with breathing)? If the patient does *not* have any pain complaint, then mark 'explicitly absent.'" \
  --var "back_pain:presence:Does the note indicate the patient is complaining about back pain?" \
  --var "cough:presence:Does the note indicate the patient is complaining of cough?" \
  --var "hemoptysis_present:presence:Look for any mention of the patient coughing up blood. Inc